In [ ]:
import os, json

aug_dicts = {}
for dataset_name in ['air_quality', 'nicu', 'synthetic']:
    read_path = f"../../data/{dataset_name}/aug_text.json"
    aug_dicts[dataset_name] = json.load(open(read_path, "r"))



In [ ]:

for dataset_name in aug_dicts.keys():
    for col in aug_dicts[dataset_name].keys():
        for unique_str in aug_dicts[dataset_name][col].keys():
            print(unique_str)
            break
        break
    break

In [ ]:
import pandas as pd

rows = []
for dataset_name in aug_dicts.keys():
    for col in aug_dicts[dataset_name].keys():
        for unique_str in aug_dicts[dataset_name][col].keys():
            value = aug_dicts[dataset_name][col][unique_str]
            rows.append({
                "dataset_name": dataset_name,
                "col": col,
                "unique_str": unique_str,
                "value": value
            })

df = pd.DataFrame(rows)
print(df)


In [ ]:
df

In [ ]:
import pandas as pd

attributes = {
    'Synthetic': {
        'Trend type': [
            ('no trend', "No trend."),
            ('upward linear', "The time series shows upward linear trend."),
            ('downward linear', "The time series shows downward linear trend."),
            ('upward quadratic', "The time series shows upward quadratic trend."),
            ('downward quadratic', "The time series shows downward quadratic trend.")
        ],
        'Seasonality': [
            ('no', "No seasonal pattern."),
            ('yes', "The time series exhibits a seasonal pattern.")
        ],
        'Shift in mean': [
            ('no shift', "No sharp shifts."),
            ('upward shift', "The mean of the time series shifts upwards."),
            ('downward shift', "The mean of the time series shifts downwards.")
        ],
        'Local variability': [
            ('low', "The time series exhibits low variability."),
            ('high', "The time series exhibits high variability.")
        ]
    },
    'Air quality': {
        'City': [
            ('Beijing', "This is air quality in Beijing."),
            ('London', "This is air quality in London.")
        ],
        'Season': [
            ('spring', "The season is spring."),
            ('summer', "The season is summer."),
            ('autumn', "The season is autumn."),
            ('winter', "The season is winter.")
        ]
    },
    'NICU heart rate': {
        'Variability': [
            ('low', "The time series exhibits low variability."),
            ('high', "The time series exhibits high variability.")
        ],
        'Bradycardia events': [
            ('yes', "Bradycardia events happened."),
            ('no', "No Bradycardia events.")
        ]
    }
}

rows = []
for dataset, attrs in attributes.items():
    for attr, values in attrs.items():
        for value, template in values:
            rows.append({
                "Dataset": dataset,
                "Attribute": attr,
                "Level": value,
                "Template": template
            })

df = pd.DataFrame(rows)
df.set_index(["Dataset", "Attribute", 'Level'], inplace=True)
df

In [ ]:
import pandas as pd

rows = []

# Synthetic
rows += [
    ["Synthetic", "Trend", "[no trend, upward linear, downward linear, upward quadratic, downward quadratic]", 
     "No trend. / The time series shows [upward linear, downward linear, upward quadratic, downward quadratic] trend."],
    ["Synthetic", "Seasonality", "[no, yes]", 
     "No seasonal pattern. / The time series exhibits a seasonal pattern."],
    ["Synthetic", "Shift in mean", "[no shift, upward shift, downward shift]", 
     "No sharp shifts. / The mean of the time series shifts [upwards, downwards]."],
    ["Synthetic", "Local variability", "[low, high]", 
     "The time series exhibits [low, high] variability."]
]

# Air quality
rows += [
    ["Air quality", "City", "[Beijing, London]", "This is air quality in [Beijing, London]."],
    ["Air quality", "Season", "[spring, summer, autumn, winter]", "The season is [spring, summer, autumn, winter]."]
]

# NICU heart rate
rows += [
    ["NICU heart rate", "Variability", "[low, high]", "The time series exhibits [low, high] variability."],
    ["NICU heart rate", "Bradycardia events", "[no, yes]", "[No Bradycardia events. | Bradycardia events happened.]"]
]

df = pd.DataFrame(rows, columns=["Dataset", "Attribute", "Level", "Template"])
df.set_index(["Dataset", "Attribute"], inplace=True)


def wrap_each_sentence(template):
    return r' \newline '.join([f"``{s.strip()}''" for s in template.split(' / ')])

df['Template'] = df['Template'].apply(wrap_each_sentence)
df

In [ ]:
import re
SPECIALS = {
    '&':  r'\&',
    '%':  r'\%',
    '$':  r'\$',
    '#':  r'\#',
    '_':  r'\_',
    '{':  r'\{',
    '}':  r'\}',
}

escape = lambda s: re.sub('|'.join(map(re.escape, SPECIALS)),
                          lambda m: SPECIALS[m.group()], str(s))

df_esc = df.applymap(escape)
df_esc.index = pd.MultiIndex.from_tuples(
    [(escape(ds), escape(attr)) for ds, attr in df.index],
    names=df.index.names)

latex_tabular = df_esc.to_latex(
    multicolumn=True,
    multirow=True,
    escape=False,
    column_format='llp{6cm}p{9cm}'   # widths still control line‑wrapping inside cells
).rstrip()                           # trim trailing newline so \resizebox ends cleanly

# replace this  &  & Level & Template \\ Dataset & Attribute &  &  \\ 
# with 
# Level & Template & Dataset & Attribute\\
latex_tabular = re.sub(
    r"&\s*&\s*Level\s*&\s*Template\s*\\\\\s*Dataset\s*&\s*Attribute\s*&\s*&\s*\\\\",
    r"Dataset & Attribute & Levels & Templates\\\\",
    latex_tabular
)
latex_tabular = re.sub(r'(\\cline\{1-4\})(?![\s\S]*\\cline\{1-4\})', '', latex_tabular)
latex_table = rf"""
\begin{{table*}}[htbp]
\centering
\resizebox{{\textwidth}}{{!}}{{%
{latex_tabular}}}
\caption{{Natural language templates for each attribute level.}}
\label{{tab:attr_template}}
\end{{table*}}
"""

print(latex_table)